In [ ]:
from datasets import load_dataset
from src.tokenizers.BPE import BytePairEncoding
from src.utils import paths

In [ ]:
tokenizer = BytePairEncoding.from_file(paths.EXPERIMENTS_DIR / "classla_finewebedu_2gb_tokenizer")

In [ ]:


hf_dataset = load_dataset(
    "wikimedia/wikipedia", 
    "20231101.hr", 
    cache_dir=paths.DATA_DIR / "wikimedia/wikipedia/20231101/hr",
)


In [ ]:
words = []
tokens = []

i = 1000
for sample in hf_dataset["train"]:
    i -= 1
    if i == 0:
        break
    words.append(sample["text"])
    tokens.append(tokenizer.tokenize(sample["text"]))

In [ ]:
print(len(tokens[0]))
print(len(words[0]))

In [ ]:
import matplotlib.pyplot as plt

# Compute the lengths of the texts (in characters) and their tokenizations (number of tokens)
text_lengths = [len(t) for t in words]
token_lengths = [len(tok) for tok in tokens]

# Compute average compression ratio: average number of characters per token
# (i.e., average compression ratio = average(original length / tokenized length))
compression_ratios = [tlen / tkl if tkl > 0 else 0 for tlen, tkl in zip(text_lengths, token_lengths)]
avg_compression_ratio = sum(compression_ratios) / len(compression_ratios)
print(f"Average compression ratio (characters per token): {avg_compression_ratio:.3f}")

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
plt.hist(text_lengths, bins=200, color="skyblue", edgecolor="black", log=True)
plt.title("Distribution of Text Lengths (characters)")
plt.xlabel("Text length (characters)")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.subplot(1,2,2)
plt.hist(token_lengths, bins=200, color="orange", edgecolor="black", log=True)
plt.title("Distribution of Tokenized Lengths (tokens)")
plt.xlabel("Tokenized length (tokens)")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
words = []
tokens = []

i = 0
for sample in hf_dataset["train"]:
    i += 1
    if i % 100 == 0:
        print(i)
    if i == 10000 :
        break
    words.extend([word for word in sample["text"].split()])

print(len(words))
words = set(words)
print(len(words))

for word in words:
    tokens.append(tokenizer.tokenize(word))


In [ ]:
import matplotlib.pyplot as plt

# Compute the number of tokens required for each word
word_token_lengths = [len(toks) for toks in tokens]

plt.figure(figsize=(8,5))
counts, bins, patches = plt.hist(
    word_token_lengths, 
    bins=range(1, max(word_token_lengths) + 2), 
    color='cadetblue', 
    edgecolor='black',
    density=True
)

# Convert density to percentage by multiplying by bin width and 100
bin_widths = [bins[i+1] - bins[i] for i in range(len(bins)-1)]
percentages = counts * bin_widths * 100

# Clear the existing plot and plot percentages
plt.clf()
plt.bar(bins[:-1], percentages, width=bin_widths, color='cadetblue', edgecolor='black', align='edge')

plt.title("Distribution of Tokenized Word Lengths")
plt.xlabel("Number of tokens per word")
plt.ylabel("Percentage (%)")
plt.xticks(range(1, max(word_token_lengths) + 1))
plt.show()

# Print percentage of "continuing words" (words that are split into more than one token)
n_words = len(word_token_lengths)
n_continuing = sum(1 for n in word_token_lengths if n > 1)
perc_continuing = (n_continuing / n_words) * 100 if n_words > 0 else 0
print(f"Percentage of continuing words (split into >1 token): {perc_continuing:.2f}%")